In [1]:
import pandas as pd

# Load the cleaned orders from notebook 1
df = pd.read_csv("data/clean_orders.csv", parse_dates=["InvoiceDate"])

# Step 1: set the cutoff = 90 days before the last order in the data
end_date = df["InvoiceDate"].max()
cutoff = end_date - pd.Timedelta(days=90)
print("Cutoff date:", cutoff)

# Step 2: split orders into "past" (before cutoff) and "future" (after)
past = df[df["InvoiceDate"] < cutoff]
future = df[df["InvoiceDate"] >= cutoff]

# Step 3: build features per customer, using only the past
g = past.groupby("Customer ID")
features = pd.DataFrame({
    "recency_days": (cutoff - g["InvoiceDate"].max()).dt.days,  # days since last order
    "frequency": g["Invoice"].nunique(),                        # number of orders
    "monetary": g["Amount"].sum(),                              # total money spent
    "tenure_days": (cutoff - g["InvoiceDate"].min()).dt.days,   # days since first order
    "total_items": g["Quantity"].sum(),                         # items bought
})
features["avg_order_value"] = features["monetary"] / features["frequency"]

# Step 4: churn label = 1 if the customer did NOT order in the future period
bought_later = set(future["Customer ID"])
features["churn"] = (~features.index.isin(bought_later)).astype(int)

# Step 5: check the result and save it
print(features.shape)
print(features["churn"].value_counts())
print(features["churn"].value_counts(normalize=True))
features.to_csv("data/customer_features.csv")
features.head()

Cutoff date: 2011-09-10 12:50:00
(3370, 7)
churn
0    1921
1    1449
Name: count, dtype: int64
churn
0    0.57003
1    0.42997
Name: proportion, dtype: float64


,recency_days,frequency,monetary,tenure_days,total_items,avg_order_value,churn
Customer ID,,,,,,,
12346,235,1,77183.60,235,74215,77183.600000,1
12347,39,5,2790.86,276,1590,558.172000,0
12348,158,3,1487.24,267,2124,495.746667,0
12350,219,1,334.40,219,197,334.400000,1
12352,171,5,1561.81,206,254,312.362000,0


In [2]:
# How many customers, and how many churned?
print(features.shape)
print(features["churn"].value_counts())
print(features["churn"].value_counts(normalize=True))

# Spot extreme values and impossible numbers
features.describe()

(3370, 7)
churn
0    1921
1    1449
Name: count, dtype: int64
churn
0    0.57003
1    0.42997
Name: proportion, dtype: float64


,recency_days,frequency,monetary,tenure_days,total_items,avg_order_value,churn
count,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000,3370.000000
mean,94.485757,3.532938,1618.814594,182.397033,952.354303,398.486264,0.429970
std,79.982942,5.788414,6151.394333,80.468646,3829.456551,1422.696074,0.495145
min,0.000000,1.000000,2.900000,0.000000,1.000000,2.900000,0.000000
25%,25.000000,1.000000,266.100000,123.000000,132.000000,169.577500,0.000000
50%,74.000000,2.000000,565.960000,193.000000,314.000000,284.140000,0.000000
75%,151.000000,4.000000,1388.937500,264.000000,783.750000,414.923125,1.000000
max,283.000000,132.000000,178302.620000,283.000000,127744.000000,77183.600000,1.000000


In [3]:
# Extra features, built from the past only
g = past.groupby("Customer ID")

# Orders in the 90 days just before the cutoff (recent activity)
recent = past[past["InvoiceDate"] >= cutoff - pd.Timedelta(days=90)]
features["orders_last_90d"] = recent.groupby("Customer ID")["Invoice"].nunique()
features["orders_last_90d"] = features["orders_last_90d"].fillna(0)  # no recent orders = 0

# How many different months the customer ordered in (regularity)
past_months = past.assign(month=past["InvoiceDate"].dt.to_period("M"))
features["active_months"] = past_months.groupby("Customer ID")["month"].nunique()

# How many different products they bought
features["unique_products"] = g["StockCode"].nunique()

# Save as a NEW file so the baseline stays untouched
features.to_csv("data/customer_features_v2.csv")
features.head()

,recency_days,frequency,monetary,tenure_days,total_items,avg_order_value,churn,orders_last_90d,active_months,unique_products
Customer ID,,,,,,,,,,
12346,235,1,77183.60,235,74215,77183.600000,1,0.0,1,1
12347,39,5,2790.86,276,1590,558.172000,0,1.0,5,82
12348,158,3,1487.24,267,2124,495.746667,0,0.0,3,22
12350,219,1,334.40,219,197,334.400000,1,0.0,1,17
12352,171,5,1561.81,206,254,312.362000,0,0.0,2,26
